# render

> Message and dialog rendering: one projection from `Message` to semantic fasttags, shared by every app that displays a dialog

In [ ]:
#| default_exp render

#| export
Every app that displays a dialog reimplements the same interpretation: dispatch on `msg_type`, strip tool traffic from replies, render outputs with `application/javascript` dropped and LaTeX normalized, inline attachments, skip pinned messages. This module holds that interpretation once, as composable pieces (`md2ft`, `render_message`) plus `__ft__` patches so a `Message` or `Dialog` renders directly in fasttag contexts. Styling stays with each consumer: output is semantic, unstyled tags carrying `data-type`/`data-part` attributes as the stable styling contract. Requires `mdhtml` (the `render` extra).

In [ ]:
#| export
import base64, json
from fastcore.utils import *
from fastcore.xml import Div, Article, Header, Main, Pre, Script, Link, Img, NotStr, to_xml, FT
from fastcore.nbio import join_out
from html import escape
from mdhtml import to_mdhtml, to_html, theme_css, math_js
from aidialog.dialog import Dialog, Message, render_md, normalize_text_latex, snote, scode, sprompt, sraw
from aidialog.msg_parts import conv_tools, strip_tools

In [ ]:
#| hide
from fastcore.test import *
from aidialog.ipynb import reads_ipynb

## Markdown to fasttags

`md2ft` is the markdown workhorse: mdhtml dialect to HTML, reference resolution, auto heading ids (which give share pages their table of contents anchors), and a per-message footnote salt so two messages' footnotes cannot collide on one page. Its `atts` parameter inlines pasted images as data URIs, so a rendered dialog is one self-contained document. `math_mode` parses the dialog-level `use_katex` setting; `katex_hdrs` returns the page headers that setting needs, with the CDN pin as a module constant an app can override.

In [ ]:
#| export
_math_names = ('off','brackets','dollars')

def math_mode(use_katex):
    "0 for off or unparseable, 1 for truthy, 2 for 'd…' (dollar delimiters)"
    try: return 2 if isinstance(use_katex,str) and use_katex[:1].lower()=='d' else int(str2bool(use_katex))
    except TypeError: return 0

def md2ft(
    md,  # Markdown source (the `md` dialect)
    atts=(),  # `Attachment`s to inline as data URIs
    math_mode:int=0,  # 0 off, 1 bracket delimiters, 2 dollars
    fn_salt=''  # Salt for footnote ids, normally the message id
):
    "Convert markdown to fasttags"
    atts = {a.id:a for a in atts}
    def embed_attachment(n, html):
        if not n['url'].startswith('attachment:'): return
        if not (a := atts.get(n['url'].removeprefix('attachment:'))): return
        data = base64.b64encode(a.data).decode()
        return to_xml(Img(src=f'data:{a.content_type};base64,{data}', alt=n['alt'] or '', title=n.get('title')))
    return NotStr(str(to_html(to_mdhtml(md, math=_math_names[math_mode], callbacks=dict(image=embed_attachment)),
                              refs='ids', auto_ids=True, fn_salt=fn_salt)))

In [ ]:
str(md2ft('# Hi\nSome *md* with $x^2$', math_mode=2))

In [ ]:
#| export
katex_cdn = 'https://cdn.jsdelivr.net/npm/katex@0.16.22'

def katex_hdrs(math_mode):
    "KaTeX stylesheet and render script, or empty when math is off"
    if not math_mode: return ()
    js = f"import katex from '{katex_cdn}/dist/katex.mjs';\n" + math_js(fn='render', minRuleThickness=0.06) + '\nrender(document);'
    return Link(rel='stylesheet', href=f'{katex_cdn}/dist/katex.min.css'), Script(js, type='module')

## The message dispatch

`render_message` is the interpretation layer: what each `msg_type` means when displayed. Notes and prompts are markdown; raw messages are verbatim; code is highlighted source plus outputs. Replies pass through `conv_tools` then `strip_tools`, so tool traffic never reaches a rendered page. `_prep_out` drops `application/javascript` outputs and normalizes `text/latex` for KaTeX. Outputs render with math forced on even when the dialog has it off, because library-emitted LaTeX arrives bracket-delimited regardless of how the author writes prose.

Math is a dialog-level setting: the app stamps `math_mode` on the `Dialog` once (from its `use_katex` config), and everything here reads it through the `dlg` backref, so no function sees a request or a database.

In [ ]:
#| export
def _hl(s): return NotStr(str(to_html(f'<pre><code class="language-python">{escape(s)}</code></pre>')))
def _mm(m): return getattr(m.dlg, 'math_mode', 0)

def _prep_out(o):
    if not (d := o.get('data')): return o
    d = {k:v for k,v in d.items() if k!='application/javascript'}
    if 'text/latex' in d: d['text/latex'] = normalize_text_latex(join_out(d['text/latex']))
    return {**o, 'data': d}

def _outputs(m):
    md = render_md([_prep_out(o) for o in m.output or []])
    return Div(md2ft(md, math_mode=_mm(m) or 1, fn_salt=m.id), data_part='output') if md else ''

def _msg_input(m):
    if m.msg_type in (snote,sprompt): return md2ft(m.content, m.attachments, _mm(m), fn_salt=m.id)
    if m.msg_type==sraw:              return Pre(m.content)
    if m.msg_type==scode:             return _hl(m.content)
    raise NotImplementedError(m.msg_type)

def _msg_output(m):
    if m.msg_type==scode: return _outputs(m)
    if m.msg_type==sprompt and (resp := m.ai_res): return md2ft(strip_tools(conv_tools(resp)), math_mode=_mm(m), fn_salt=m.id)
    return ''

def render_message(m):
    "One message as fasttags: input plus output, shaped by `msg_type`"
    inp,out = _msg_input(m),_msg_output(m)
    if m.msg_type==sprompt: return Header(inp), Div(out, data_part='reply') if out else ''
    return (inp,out) if out else inp

In [ ]:
#| export
@patch
def __ft__(self:Message): return Article(render_message(self), id=f'_{self.id}', data_type=self.msg_type)

@patch
def __ft__(self:Dialog): return Main(m for m in self.messages if not m.meta.get('pinned'))

The `__ft__` patches make a `Message` or whole `Dialog` render directly in any fasttag context (`to_xml(dlg)`, or returning a dialog from a FastHTML route). The article id gets a `_` prefix because message ids can start with a digit, which HTML ids cannot; `data_type` and `data-part` are the stable selectors consumer stylesheets target.

## Rendering a dialog

A three-message dialog exercises every path: a note with math, a code message with an output, and a prompt whose stored reply carries a tool fence. (A prompt in ipynb form is a markdown cell with `solveit_ai` metadata and the reply inline after the separator line.)

In [ ]:
_toy = {'cells':[
  {'cell_type':'markdown','id':'aaaa1111','metadata':{},
   'source':'# Fourier for felines\nA *tiny* demo with math: $e^{i\\pi}+1=0$'},
  {'cell_type':'code','id':'bbbb2222','metadata':{},'execution_count':1,'source':'6*7',
   'outputs':[{'output_type':'execute_result','data':{'text/plain':['42']},'metadata':{},'execution_count':1}]},
  {'cell_type':'markdown','id':'cccc3333','metadata':{'solveit_ai':True},
   'source':'Why is the answer 42?\n\n##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->\n\nBecause Douglas Adams said so.\n\n```json {.tool}\n{"id":"t1","name":"search","args":{"q":"42"},"result":"..."}\n```\n\nNo further questions.'},
 ],'metadata':{},'nbformat':4,'nbformat_minor':5}
dlg = reads_ipynb(json.dumps(_toy))
dlg.math_mode = 2
print(to_xml(dlg.messages[2]))

The prompt rendered as a `header` (the question) plus a `data-part="reply"` div, and the tool fence is gone. Each check below pins a behavior a future mdhtml or aidialog change could silently break: dollar math becomes KaTeX spans, code highlights server-side with its output present, replies keep their structure, and tool traffic never leaks.

In [ ]:
note_html,code_html,prompt_html = (to_xml(m) for m in dlg.messages)
assert 'math inline' in note_html
assert 'hl-number' in code_html and '42' in code_html
assert 'data-part="reply"' in prompt_html
assert '{.tool}' not in prompt_html and 'search' not in prompt_html
to_xml(dlg)[:120]

A pinned message disappears from the dialog view, and `katex_hdrs` returns headers exactly when math is on:

In [ ]:
dlg.messages[0].meta['pinned'] = True
assert 'felines' not in to_xml(dlg)
assert katex_hdrs(0)==() and len(katex_hdrs(2))==2
test_eq(math_mode('dollars'), 2); test_eq(math_mode(''), 0); test_eq(math_mode(True), 1); test_eq(math_mode(None), 0)
'all checks pass' 